# convT-as-flipped-padded-conv — worked example 3: The same identity in 1D: ConvTranspose1d as flipped padded Conv1d

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-as-flipped-padded-conv`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The flip-and-pad identity is dimension-agnostic. For `F.conv_transpose1d(x, weight)` (stride 1, no padding) with `x: (B, IC, L)` and `weight: (IC, OC, K)`, the equivalent is `F.conv1d` on `x` padded by `K-1` on both ends, with the kernel flipped along its single spatial axis and channel axes swapped. Working it in 1D strips away the second spatial dimension and makes the single flip axis obvious.

## Worked solution

We reproduce `F.conv_transpose1d(x, weight)` using only `F.conv1d`.

**Step 1 - pad by `K-1` on both ends.** With a length-`K` kernel, ConvT1d outputs length `L + K - 1`. Valid conv1d shrinks by `K-1`, so padding the input by `K-1` on each end (`F.pad(x, (K-1, K-1))` - note 1D pad takes a 2-tuple `(left, right)`) makes conv1d output the right length.

**Step 2 - flip the kernel along the lone spatial axis.** `weight.flip(-1)` reverses the `K` taps. In 1D there is only one axis to flip, which makes the adjoint structure especially clear: the transpose operation reads the kernel backwards.

**Step 3 - swap channel axes.** ConvT1d weight is `(IC, OC, K)`; conv1d wants `(OC, IC, K)`. `.transpose(0, 1)` swaps them.

**Step 4 - run conv1d and compare.** `F.conv1d(x_pad, w)` matches the ConvT1d reference to floating-point tolerance, confirmed with `torch.allclose`.

Seeing the identity collapse to a single `flip(-1)` in 1D makes it memorable: 'pad by K-1, reverse the kernel, swap channels' is the whole trick, and the 2D version just applies the flip on every spatial axis.

In [ ]:
import torch.nn.functional as F

def convT1d_as_padded_conv(x, weight):
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1))           # (B, IC, L+2(K-1))
    w = weight.flip(-1).transpose(0, 1)        # reverse taps + (IC,OC)->(OC,IC)
    return F.conv1d(x_pad, w)

t.manual_seed(0)
x = t.randn(2, 3, 7)         # (B=2, IC=3, L=7)
weight = t.randn(3, 5, 4)    # ConvT1d layout (IC=3, OC=5, K=4)

rebuilt = convT1d_as_padded_conv(x, weight)
reference = F.conv_transpose1d(x, weight)
print('rebuilt shape:', tuple(rebuilt.shape))   # expect (2, 5, 10)
print('matches ConvT1d:', t.allclose(rebuilt, reference, atol=1e-5))
print('max abs diff:', (rebuilt - reference).abs().max().item())